# Mini DDPM CIFAR-10 — Training di Colab

Kelompok 14. Jalankan cell dari atas ke bawah. Pastikan **Runtime → Change runtime type → GPU (T4)**.

Checkpoint (`last.pt`) dan `loss_log.csv` disimpan ke Google Drive. Bila Colab terputus, jalankan ulang
cell setup lalu cell training yang sama: training otomatis lanjut (resume) dari epoch terakhir.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
%cd /content
!git clone https://github.com/ratrifa/mini-ddpm-cifar10.git || git -C mini-ddpm-cifar10 pull
%cd /content/mini-ddpm-cifar10

## 1. Uji coba training singkat

2 epoch × 200 langkah. Cek: loss turun dan `last.pt` muncul di Drive.

In [ ]:
TRIAL_DIR = "/content/drive/MyDrive/mini-ddpm/trial"
!python src/train.py --epochs 2 --steps-per-epoch 200 --data-root /content/data --ckpt-dir {TRIAL_DIR}
!ls -la {TRIAL_DIR}
!cat {TRIAL_DIR}/loss_log.csv

## 2. Training penuh

Ganti `RUN_DIR` untuk tiap eksperimen (Sesi 8) agar checkpoint tidak tertimpa. Hyperparameter diubah lewat
argumen, misalnya `--lr 1e-4`, `--batch-size 64`, `--timesteps 500`, `--base-ch 96`.

In [ ]:
RUN_DIR = "/content/drive/MyDrive/mini-ddpm/run1"
!python src/train.py --epochs 100 --data-root /content/data --ckpt-dir {RUN_DIR}

## 3. Sampling (reverse process)

Membuat 8 gambar per kelas dari bobot EMA di checkpoint, untuk 4 guidance scale sekaligus: 0 (tanpa kelas),
1 (conditional biasa), 3 (default), dan 5. Tiap baris = satu kelas (airplane, automobile, bird, cat, deer, dog,
frog, horse, ship, truck). Seed sama untuk tiap skala, jadi perbedaan antar-grid murni dari guidance.

Hasil disimpan ke folder run di Drive: `samples_g0.png` … `samples_g5.png` plus `.pt` (tensor gambar + label
untuk `visualize.show_sample_grid`). 1000 langkah reverse per skala, jadi butuh beberapa menit per skala.
Ubah `RUN` ke folder `trial` untuk mengecek hasil uji coba singkat (hasilnya masih noise, itu wajar).

In [ ]:
RUN = "/content/drive/MyDrive/mini-ddpm/run1"  # bisa diganti ke folder trial / eksperimen lain
SCALES = [0, 1, 3, 5]
scales_arg = " ".join(str(s) for s in SCALES)
!python src/sample.py --ckpt {RUN}/last.pt --out {RUN}/samples.png --guidance-scales {scales_arg}

from IPython.display import Image, display
for s in SCALES:
    print(f"guidance {s}")
    display(Image(f"{RUN}/samples_g{s}.png", width=400))